In [20]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import sys
import datetime as dt
from scipy.interpolate import interp1d
from pyproj import Proj

In [21]:
%matplotlib inline

In [22]:
sys.path.append('/Users/Admin/Documents/scripts/pyfvcom')
import PyFVCOM as pf

In [4]:
def write_river_namelist(output_file, conf_dict):
    """
    Write an FVCOM river namelist file.

    Parameters
    ----------
    output_file : str, pathlib.Path
        Output file to which to write the river configuration.
    forcing_file : str, pathlib.Path
        File from which FVCOM will read the river forcing data.
    vertical_distribution : str, optional
        Vertical distribution of river input. Defaults to 'uniform'.

    """
    for ri in np.arange(0,len(conf_dict['RIVER_GRID_LOCATION'])):
        namelist = {'NML_RIVER': [pf.preproc.NameListEntry('RIVER_NAME', conf_dict['RIVER_NAME'][ri]),
                                  pf.preproc.NameListEntry('RIVER_FILE', conf_dict['RIVER_FILE']),
                                  pf.preproc.NameListEntry('RIVER_GRID_LOCATION', conf_dict['RIVER_GRID_LOCATION'][ri] + 1, 'd'),
                                  pf.preproc.NameListEntry('RIVER_VERTICAL_DISTRIBUTION', conf_dict['RIVER_VERTICAL_DISTRIBUTION'][ri])]}
        pf.preproc.write_model_namelist(output_file, namelist, mode='a')


In [5]:
# 70°24.907 N, 26°42.900 Ø
# 70.41512 N, 26.715 E

In [6]:
# lon, lat, Adamselv
pipe_loc = (26.715, 70.415)

In [7]:
pipe_diameter = 1  # example
pipe_depth = 33  # actual depth

outflow_deps = [pipe_depth - pipe_diameter/2, pipe_depth + pipe_diameter/2]
# outflow_deps: [48.5, 51.5]

# Make outflow

In [8]:
5.3/60

0.08833333333333333

In [9]:
start_date = dt.datetime(2013,2,15)
release_length = dt.timedelta(days=90)
#total_release = 5.3 / 60  # UNITS?

end_release = start_date + release_length
end_run = dt.datetime(2013,10,30)

# Why do we * here??
#outflow = total_release*(release_length/dt.timedelta(hours=1))  # Convert to m3/hr
#outflow_m3_per_sec = outflow/3600 # convert m3/hr to m3/sec

In [10]:
outflow_m3_per_sec = 5.3 / 60

In [11]:
# NB! not really good one, as it crashed
out_file = xr.open_dataset('adamselv_v01_0001.nc', decode_times=False)  

In [12]:
node_sig_deps = out_file['siglay'].values * out_file['h'].values[np.newaxis,:]

x = out_file['x'].values
y = out_file['y'].values
xc = out_file['xc'].values
yc = out_file['yc'].values
tri = np.asarray(out_file['nv'][:] - 1).T

In [13]:
utm33 = Proj(proj = 'utm', zone = 33, ellps = 'WGS84', preserve_units = False)
pl = utm33(pipe_loc[0], pipe_loc[1]) 

pipe_nodes = np.array(np.argmin(np.sqrt((x - pl[0])**2 + (y - pl[1])**2)))
pipe_deps = node_sig_deps[:,pipe_nodes].T

In [ ]:
70.41748 Ø: 26.70989

In [24]:
l = utm33(26.70989, 70.41748) 
np.array(np.argmin(np.sqrt((x - l[0])**2 + (y - l[1])**2)))

array(52557)

In [14]:
outflow_deps 

[32.5, 33.5]

In [15]:
pipe_nodes # COPY TO NML!!

array(54469)

In [16]:
pipe_dep_split = []

choose = np.array(np.logical_and(pipe_deps >= -outflow_deps[1], pipe_deps <= -outflow_deps[0]))
print(choose)
this_split = np.zeros(len(choose))
this_split[choose] = 1/np.sum(choose)
pipe_dep_split.append(this_split)

pipe_dep_split = np.asarray(pipe_dep_split)
pipe_dep_split_str = []

for this_pipe in pipe_dep_split:
    this_str = ''
    for this_out in this_pipe:
        this_str = this_str + f'{this_out} '
    pipe_dep_split_str.append(this_str[0:-1])

# Generate an array of datetime values at 3-hour intervals
pipe_dt = np.array([
    start_date + dt.timedelta(hours=i)
    for i in range(0, int((end_run - start_date).total_seconds() / 3600) + 1, 3)
])

# Make flux
pipe_flux = np.asarray(np.ones([pipe_nodes.size, pipe_dt.size])*(outflow_m3_per_sec/pipe_nodes.size)).T


[False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False  True False False]


In [17]:
pipe_dep_split  # COPY TO NML!

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])

**!! Comment: then I just went to the original NML file and manually added the pipe river**

# Add river (pipe)

In [18]:
pipe_c = 10
pipe_flux = 5.3/60  # m/s
pipe_temp = 9.6
pipe_salt = 9
pipe_name = "Pipe"

In [19]:
5.3/60

0.08833333333333333

In [21]:
# open riverdata ds
ds = xr.open_dataset('../02_build_rivers/riverdata.nc', decode_times=False)

In [22]:
nt = ds.sizes['time']
nrivers = ds.sizes['rivers']

In [23]:
pipe_flux_array = np.full((nt, 1), pipe_flux)
pipe_flux_array[:112] = 0

In [24]:
# Create a new dataset with the same 'time' dimension but 'rivers' = 1
pipe_ds = xr.Dataset(
    {
        "river_flux": (["time", "rivers"], np.full((nt, 1), pipe_flux_array)),
        "river_temp": (["time", "rivers"], np.full((nt, 1), pipe_temp)),
        "river_salt": (["time", "rivers"], np.full((nt, 1), pipe_salt)),
        "river_names": (["rivers"], np.array([pipe_name], dtype=object))
    },
    coords={
        "time": ds["time"].values,
        # "rivers": np.array([234])  # Single river index = length of the original rivers
    },
    attrs=ds.attrs  # Copy attributes from the original dataset
)

In [25]:
out = xr.concat([ds, pipe_ds], 'rivers', data_vars='minimal')

In [137]:
# write an intermediate result just to test run with a new river (without FABM tracer)
out.to_netcdf('riverdata_pipe.nc')

# Add tracer river

In [26]:
# just copy the variable
out['tracer1_c'] = out['river_salt']*0
out['tracer2_c'] = out['river_salt']*0

In [27]:
abs(56324 - 56324.125)*24

3.0

In [28]:
8 * 14  # start index time for discharge (we start discharge after 2 weeks)

112

In [29]:
tracer1_c_data = np.zeros(out['river_salt'].shape)
# fill the last river with constant
tracer1_c_data[112:,-1] = 10

# dump to the dataset
out['tracer1_c'].values = tracer1_c_data


tracer2_c_data = np.zeros(out['river_salt'].shape)
# fill the last river with constant
tracer2_c_data[112:,-1] = 10

# dump to the dataset
out['tracer2_c'].values = tracer2_c_data

In [30]:
out.to_netcdf('riverdata_pipe_tracer.nc')

# Add tracer to Nest

In [99]:
nest = xr.open_dataset('fvcom_nest_adamselv.nc', decode_times=False)

In [70]:
nest

<xarray.Dataset> Size: 245MB
Dimensions:        (time: 3577, node: 112, nele: 110, three: 3, siglay: 30,
                    siglev: 31)
Coordinates:
  * time           (time) float32 14kB 5.632e+04 5.632e+04 ... 5.647e+04
    siglay         (siglay, node) float32 13kB ...
    siglev         (siglev, node) float32 14kB ...
Dimensions without coordinates: node, nele, three
Data variables: (12/23)
    Itime          (time) int32 14kB ...
    Itime2         (time) int32 14kB ...
    lon            (node) float32 448B ...
    lat            (node) float32 448B ...
    lonc           (nele) float32 440B ...
    latc           (nele) float32 440B ...
    ...             ...
    va             (time, nele) float32 2MB ...
    u              (time, siglay, nele) float32 47MB ...
    v              (time, siglay, nele) float32 47MB ...
    temp           (time, siglay, node) float32 48MB ...
    salinity       (time, siglay, node) float32 48MB ...
    hyw            (time, siglev, node) float32 50MB ...

In [76]:
plt.scatter(nest.x, nest.y)
plt.savefig('nest.png')

In [100]:
nest['tracer1_c'] = nest['temp']*0 
nest['tracer1_c_bot'] = nest['zeta']*0 

nest['tracer2_c'] = nest['temp']*0 
nest['tracer2_c_bot'] = nest['zeta']*0 

In [101]:
nest.to_netcdf('fvcom_nest_adamselv_tracer.nc')

# Restart

In [102]:
rst = xr.open_dataset('adamselv_v01_restart_0001.nc', decode_times=False)

In [103]:
rst['tracer1_c'] = rst['temp']*0
rst['tracer1_c_bot'] = rst['et']*0 

rst['tracer2_c'] = rst['temp']*0
rst['tracer2_c_bot'] = rst['et']*0 

In [89]:
rst.temp.min()

<xarray.DataArray 'temp' ()> Size: 8B
array(0.76934159)

In [104]:
rst.to_netcdf('adamselv_v01_restart_0001_tracer.nc')